In [4]:
import numpy as np

def tanh(x):
    return np.tanh(x)

def softmax(x):
    exp_x = np.exp(x - np.max(x))  
    return exp_x / np.sum(exp_x)

class WordRNN:
    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size
        self.Wx = np.random.randn(hidden_size, input_size) * 0.1  
        self.Wh = np.random.randn(hidden_size, hidden_size) * 0.1  
        self.Wy = np.random.randn(output_size, hidden_size) * 0.1 
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))

    def forward(self, inputs, h_prev):
        self.inputs = inputs
        self.hs = [h_prev]
        self.as_ = []
        self.ys = []

        for x in inputs:
            a_t = np.dot(self.Wx, x) + np.dot(self.Wh, h_prev) + self.bh
            h_t = tanh(a_t)
            y_t = softmax(np.dot(self.Wy, h_t) + self.by)

            self.as_.append(a_t)
            self.hs.append(h_t)
            self.ys.append(y_t)
            h_prev = h_t

        return self.ys, self.hs[-1]

    def backward(self, targets, learning_rate=0.01):
        dWx = np.zeros_like(self.Wx)
        dWh = np.zeros_like(self.Wh)
        dWy = np.zeros_like(self.Wy)
        dbh = np.zeros_like(self.bh)
        dby = np.zeros_like(self.by)
        dh_next = np.zeros((self.hidden_size, 1))

        loss = 0
        for t in reversed(range(len(targets))):
            y_pred = self.ys[t]
            target = targets[t]
            loss += -np.sum(target * np.log(y_pred + 1e-8))

            dy = y_pred - target
            dWy += np.dot(dy, self.hs[t+1].T)
            dby += dy

            dh = np.dot(self.Wy.T, dy) + dh_next
            da = dh * (1 - self.hs[t+1]**2)

            dWx += np.dot(da, self.inputs[t].T)
            dWh += np.dot(da, self.hs[t].T)
            dbh += da

            dh_next = np.dot(self.Wh.T, da)

        self.Wx -= learning_rate * dWx
        self.Wh -= learning_rate * dWh
        self.Wy -= learning_rate * dWy
        self.bh -= learning_rate * dbh
        self.by -= learning_rate * dby

        return loss

def prepare_data():
    print("Please enter a 4-word text (e.g., 'I like to run'):")
    text = input().strip()
    words = text.split()

    if len(words) != 4:
        raise ValueError("Input must contain exactly 4 words.")

    unique_words = sorted(set(words))  
    vocab_size = len(unique_words)
    word_to_idx = {word: i for i, word in enumerate(unique_words)}
    idx_to_word = {i: word for i, word in enumerate(unique_words)}

    def one_hot_encode(word, vocab_size):
        vec = np.zeros((vocab_size, 1))
        vec[word_to_idx[word]] = 1
        return vec

    inputs = [one_hot_encode(word, vocab_size) for word in words[:3]] 
    targets = [one_hot_encode(word, vocab_size) for word in words[1:]]

    return inputs, targets, vocab_size, word_to_idx, idx_to_word, words

try:
    inputs, targets, vocab_size, word_to_idx, idx_to_word, original_words = prepare_data()

    hidden_size = 3  
    rnn = WordRNN(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)
    h_prev = np.zeros((hidden_size, 1))

    epochs = 1000
    for epoch in range(epochs):
        outputs, h_prev = rnn.forward(inputs, h_prev)
        loss = rnn.backward(targets)
        if epoch % 100 == 0:
            print(f'Epoch {epoch}, Loss: {loss:.4f}')

    h_prev = np.zeros((hidden_size, 1))
    test_input = [np.zeros((vocab_size, 1)) for _ in range(3)]
    for i, word in enumerate(original_words[:3]):
        test_input[i] = np.zeros((vocab_size, 1))
        test_input[i][word_to_idx[word]] = 1

    outputs, h_prev = rnn.forward(test_input, h_prev)
    predicted_idx = np.argmax(outputs[-1])
    predicted_word = idx_to_word[predicted_idx]

    print(f"\nInput text: {' '.join(original_words)}")
    print(f"Predicted 4th word: {predicted_word}")
    print(f"Full 4-word sequence: {' '.join(original_words[:3])} {predicted_word}")

except ValueError as e:
    print(f"Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Please enter a 4-word text (e.g., 'I like to run'):
I want to play
Epoch 0, Loss: 4.1010
Epoch 100, Loss: 3.1900
Epoch 200, Loss: 1.4479
Epoch 300, Loss: 0.6239
Epoch 400, Loss: 0.3722
Epoch 500, Loss: 0.2613
Epoch 600, Loss: 0.2002
Epoch 700, Loss: 0.1619
Epoch 800, Loss: 0.1357
Epoch 900, Loss: 0.1167

Input text: I want to play
Predicted 4th word: play
Full 4-word sequence: I want to play
